In [1]:
from src.data_loader import IoTDataLoader
from src.preprocessor import IoTPreprocessor
from src.feature_engineering import IoTFeatureEngineer
from src.models import IoTModels
from src.eda import IoTEda
from src.evaluator import IoTEvaluator
from src.balancer import IoTBalancer

loader = IoTDataLoader(file_path='data\\RT_IOT2022.csv')

raw_df = loader.load_data()


Data loaded successfully. Shape: (123117, 85)


In [2]:
eda = IoTEda()

eda.run_integrated_analysis(raw_df)


PHASE 1: FULL INTEGRATED EXPLORATORY DATA ANALYSIS

[STATS] Class Distribution:
  DOS_SYN_Hping                       94,659  ( 76.9%) ← ATTACK
  Thing_Speak                          8,108  (  6.6%) ← NORMAL
  ARP_poisioning                       7,750  (  6.3%) ← ATTACK
  MQTT_Publish                         4,146  (  3.4%) ← NORMAL
  NMAP_UDP_SCAN                        2,590  (  2.1%) ← ATTACK
  NMAP_XMAS_TREE_SCAN                  2,010  (  1.6%) ← ATTACK
  NMAP_OS_DETECTION                    2,000  (  1.6%) ← ATTACK
  NMAP_TCP_scan                        1,002  (  0.8%) ← ATTACK
  DDOS_Slowloris                         534  (  0.4%) ← ATTACK
  Wipro_bulb                             253  (  0.2%) ← NORMAL
  Metasploit_Brute_Force_SSH              37  (  0.0%) ← ATTACK
  NMAP_FIN_SCAN                           28  (  0.0%) ← ATTACK

[METADATA] Features: 83 | Missing: 0 | Duplicates: 0

[REPORT] Skewness Values:
bwd_pkts_tot             249.906300
fwd_pkts_tot             121.43591

In [4]:
prep = IoTPreprocessor()

X_train, X_test, y_train, y_test = prep.fit_transform_split(raw_df)

print(X_train.shape)

(98493, 81)


In [6]:
engineer = IoTFeatureEngineer()

X_train_fe = engineer.fit_transform(X_train)
X_test_fe  = engineer.transform(X_test)

engineer.summary()
engineer.plot(raw_df, save_path="results\\feature_engineering\\fe.png")

  FEATURE ENGINEERING SUMMARY
  Group 1 — Packet Ratios        → 5 features
  Group 2 — Payload              → 5 features
  Group 3 — IAT Timing           → 5 features
  Group 4 — TCP Flags            → 5 features
  Group 5 — Header Size          → 4 features
  Group 6 — Active/Idle          → 5 features
  Group 7 — Bulk/Subflow         → 5 features

  Total engineered features    : 34
  + Paper features             : 23
  = Final feature set          : 57
  Saved → results\feature_engineering\fe.png


In [7]:
balancer = IoTBalancer(strategy="smote")

X_train_bal, y_train_bal = balancer.fit_resample(
    X_train_fe.values,
    y_train.values
)

balancer.summary()

  BALANCING TRAINING DATA
  Before → Normal: 10,006 | Attack: 88,487
  After  → Normal: 88,487 | Attack: 88,487
  Strategy used   : SMOTE
  Total samples before : 98,493
  Total samples after  : 176,974
  Normal:Attack ratio before : 0.113
  Normal:Attack ratio after  : 1.000


In [8]:
trainer = IoTModels(save_dir="results/models")
trainer.train_all(X_train_bal, y_train_bal)

[1] Training Decision Tree …
    Saved → decision_tree.pkl
[2] Training SVM … (subsampled to 40k)
    Saved → svm.pkl
[3] Training Neural Network …
    Saved → neural_network.pkl

  ✓ All models trained and saved.


In [10]:
import pandas as pd

evaluator = IoTEvaluator(save_dir="results/evaluation")

summary = evaluator.evaluate_all(
    models = trainer.get_all_models(),
    X_test = X_test_fe.values,
    y_test = y_test.values
)


dt     = trainer.get_model("Decision Tree")
fi     = pd.Series(dt.feature_importances_,
                   index=X_train_fe.columns).sort_values(ascending=False)


nn     = trainer.get_model("Neural Network")

evaluator.plot(
    y_test             = y_test.values,
    feature_importances = fi,
    loss_curve         = nn.loss_curve_
)

print(summary.round(4))


  ── Decision Tree ──
              precision    recall  f1-score   support

      Normal       0.95      0.98      0.97      2501
      Attack       1.00      0.99      1.00     22123

    accuracy                           0.99     24624
   macro avg       0.97      0.99      0.98     24624
weighted avg       0.99      0.99      0.99     24624


  ── SVM ──
              precision    recall  f1-score   support

      Normal       0.91      0.99      0.95      2501
      Attack       1.00      0.99      0.99     22123

    accuracy                           0.99     24624
   macro avg       0.96      0.99      0.97     24624
weighted avg       0.99      0.99      0.99     24624


  ── Neural Network ──
              precision    recall  f1-score   support

      Normal       0.92      0.99      0.95      2501
      Attack       1.00      0.99      0.99     22123

    accuracy                           0.99     24624
   macro avg       0.96      0.99      0.97     24624
weighted avg  